# Analyse exploratoire — MediQAl

### Synthèse & observations

**Structure**
- Dataset : MediQAl (version OEQ)
- Langue : français
- Nombre d’exemples : 4 969
- Colonnes principales : `clinical_case`, `question`, `answer`

**Qualité des données**
- Aucun doublon exact détecté
- 20,4 % des lignes n’ont pas de `clinical_case`
- Les exemples sans cas clinique restent exploitables (questions médicales directes avec réponse)

**Contenu**
- Principaux types de questions : `Reasoning` et `Understanding`
- Incohérence mineure observée : présence de `Raisonnement`
- Corpus médical varié (pharmacie, cardiologie, pneumologie, neurologie…)

**Longueur des textes**
- Questions : longueur moyenne ~130 caractères
- Réponses : longueur moyenne ~241 caractères
- Présence d’exemples courts et plus détaillés

**Décisions de préparation**
- Conserver les exemples sans `clinical_case`
- Si un cas clinique est présent : concaténer `clinical_case + question`
- Sinon : utiliser uniquement `question`
- Conserver uniquement les champs utiles pour le SFT
- Ajouter les métadonnées `source` et `language`
- Réaliser le split train / validation / test par groupe de `clinical_case` lorsque celui-ci est disponible, afin d’éviter qu’un même cas clinique apparaisse à la fois dans l’entraînement et l’évaluation.


In [2]:
import pandas as pd
from datasets import load_dataset

mediqal_dataset = load_dataset("ANR-MALADES/MediQAl", "oeq")
mediqal_dataset

DatasetDict({
    test: Dataset({
        features: ['id', 'clinical_case', 'cc_question_number', 'question', 'answer', 'medical_subject', 'question_type'],
        num_rows: 4969
    })
})

## 1. Exploration de la structure du dataset

In [3]:
df = pd.DataFrame(mediqal_dataset["test"])

print(f"Shape : {df.shape}")
print(f"Colonnes : {list(df.columns)}")

df.head(5)

Shape : (4969, 7)
Colonnes : ['id', 'clinical_case', 'cc_question_number', 'question', 'answer', 'medical_subject', 'question_type']


,id,clinical_case,cc_question_number,question,answer,medical_subject,question_type
0,1,"Homme, 58 ans, majoration de dyspnée chez un B...",1,Quels sont éléments de gravité au moment de l’...,"FC>125bpm\n Terrain : ATCD de BPCO, insuffisan...",Emergency Medicine,Reasoning
1,2,"Homme, 58 ans, majoration de dyspnée chez un B...",2,Caractérisez en une phrase le tableau syndrom...,Insuffisance respiratoire aiguë / exacerbation...,Emergency Medicine,Reasoning
2,3,"Homme, 58 ans, majoration de dyspnée chez un B...",3,Quelle est l'explication la plus probable de l...,"Signes d’hypercapnie, pas d’effet du traitemen...",Emergency Medicine,Reasoning
3,4,"Homme, 58 ans, majoration de dyspnée chez un B...",4,Où le patient doit-il être hospitalisé ?,"En réanimation, car pas de réponse au traiteme...",Emergency Medicine,Reasoning
4,5,"M. Shoot, 30 ans.\n Agitation à la sortie d’un...",1,Quelle substance a-t-il prise ce soir ? Quels ...,"Cocaïne, qui donne un état d’agitation.\n Sign...",Emergency Medicine,Reasoning


## 2. Vérification des valeurs manquantes

In [4]:
df.isnull().sum()

id                       0
clinical_case         1015
cc_question_number    1016
question                 0
answer                   0
medical_subject          0
question_type            0
dtype: int64

## 3. Vérification des doublons

In [7]:
## Analyse des cas cliniques répétés

# On regarde combien de fois chaque cas clinique apparaît
clinical_case_counts = (
    df.dropna(subset=["clinical_case"])
      .groupby("clinical_case")
      .size()
      .sort_values(ascending=False)
)

print("Nombre de cas cliniques distincts :", clinical_case_counts.shape[0])
print("Nombre de cas cliniques avec plusieurs questions :", (clinical_case_counts > 1).sum())



Nombre de cas cliniques distincts : 1011
Nombre de cas cliniques avec plusieurs questions : 720


In [8]:
# Exemple d'un même cas clinique associé à plusieurs questions
example_case = clinical_case_counts[clinical_case_counts > 1].index[0]

df[df["clinical_case"] == example_case][
    ["clinical_case", "cc_question_number", "question", "answer"]
]

,clinical_case,cc_question_number,question,answer
3156,Un homme de 85 ans est adressé aux urgences po...,1,Quels examens complémentaires demandez-vous en...,"ECG \nRadiographie thoracique de face\nNFS, CR..."
3157,Un homme de 85 ans est adressé aux urgences po...,2,Décrivez précisément les résultats attendus de...,ECG: AC/FA\nRX Thorax: opacités alvéolaires fl...
3158,Un homme de 85 ans est adressé aux urgences po...,3,Quel diagnostic faites-vous pour les lésions d...,"Erysipèle de jambe, devant une grosse jambe ai..."
3159,Un homme de 85 ans est adressé aux urgences po...,4,Indiquez les principaux facteurs de risque de ...,"Obésité\nInsuffisance veineuse(varices, dermit..."
3160,Un homme de 85 ans est adressé aux urgences po...,5,Expliquez quelle est la relation entre la path...,La fièvre élevée liée à l'Infection cutanée a ...
3161,Un homme de 85 ans est adressé aux urgences po...,6,Quel traitement à visée cardiovasculaire presc...,Voie d'abord veineuse avec G 5% 1.5l/24h avec...
3162,Un homme de 85 ans est adressé aux urgences po...,7,Quels sont les éléments de surveillance cliniq...,"Pouls(scope), tension artérielle, fréquence re..."
3163,Un homme de 85 ans est adressé aux urgences po...,8,Quels examens paracliniques vous semblent util...,"ECG, radiographie pulmonaire, ionogramme sangu..."
3164,Un homme de 85 ans est adressé aux urgences po...,9,Quel traitement proposez-vous pour les lésions...,repos strict au lit\narceau sur la jambe gauch...
3165,Un homme de 85 ans est adressé aux urgences po...,10,Quels sont les éléments de surveillance de ce ...,Cerclage au feutre de la zone inflammatoire de...


### Risque de fuite train/test

MediQAl contient plusieurs questions associées à un même cas clinique.  
Si le split train/validation/test est fait aléatoirement ligne par ligne, certaines questions d'un même cas clinique peuvent se retrouver dans le train et d'autres dans le test.

Cela créerait une fuite de données : le modèle aurait déjà vu le contexte patient pendant l'entraînement.

Décision de préparation :
- effectuer le split MediQAl par groupe de `clinical_case` ;
- conserver toutes les questions d'un même cas clinique dans le même split ;
- appliquer cette logique avant de supprimer les colonnes `clinical_case` et `cc_question_number`.

## 4. Distribution des types de questions

In [9]:
df["question_type"].value_counts()

question_type
Reasoning        3125
Understanding    1842
Raisonnement        2
Name: count, dtype: int64

## 5. Distribution des spécialités médicales

In [10]:
df["medical_subject"].value_counts().head(10)

medical_subject
Pharmacy                     1544
Pediatric Cardiology         1225
Cardiology                    217
Nephro-Urology                193
Pulmonology                   184
Intensive Care                154
Neurology                     150
Gynecology and Obstetrics     149
Otorhinolaryngology (ENT)     126
Hematology                    124
Name: count, dtype: int64

## Analyse de la longueur des questions et réponses

In [11]:
df["question_length"] = df["question"].str.len()
df["answer_length"] = df["answer"].str.len()

df[["question_length", "answer_length"]].describe()

,question_length,answer_length
count,4969.000000,4969.000000
mean,130.039847,241.383981
std,115.882786,222.561884
min,10.000000,1.000000
25%,65.000000,56.000000
50%,98.000000,176.000000
75%,149.000000,369.000000
max,1545.000000,1020.000000


## 7. Analyse des cas cliniques manquants

In [12]:
missing_clinical = df["clinical_case"].isnull().sum()
total = len(df)

print(f"Cas cliniques manquants : {missing_clinical}")
print(f"Pourcentage : {missing_clinical / total * 100:.2f}%")

Cas cliniques manquants : 1015
Pourcentage : 20.43%


In [13]:
df[df["clinical_case"].isnull()][["question", "answer"]].head(10)

,question,answer
10,Quel est l’habitat normal des différentes espè...,"Peau , muqueuses, environnement"
11,"Chez quels patients observe t’on, le plus souv...",Infections nocosomiales sur PAC; chez l'immuno...
12,Comment peut-on établir le diagnostic d’une ca...,Hémocultures mais pas très puissant
13,"En dehors de la grossesse, dans quelles circon...",Chez l'immunodéprimé (maladie opportuniste) . ...
14,Quelle est la parasitose dont le dépistage sér...,Toxoplasmose. Le plus tôt possible
15,"Quelles sont les conséquences, cliniques et di...",Microhémorragies tissulaires
16,Quel prélèvement et quelle(s) technique(s) doi...,Goutte épaisse / frottis sanguin\n Délai maxim...
17,Citez trois parasitoses que l’on peut contract...,Tichinellose\n Toxoplasmose\n Cestodose
165,Où se différencient les lymphocytes B ?,MO
166,Quelle est la fonction essentielle des lymphoc...,"Différenciation en plasmocytes, production d’A..."
